# Forge: a short campaign

The same window and covariance as `block.ipynb`, a `Wheel` with that
covariance registered as a `FixedNoiseBlock` noise block, the block (which
declares `role = "signal"`) registered on its grid, and a chain of its
parameters collected through `on_cycle_complete`. A second block goes where the comment says. The block
starts from a prior draw, so the residual's drop over the first cycle is the
block discarding that start, not a fit: compare with the data's own rms.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np

from block import ForgeBlock
from data import analytic_covariance, load_window, white_psd
from enchilada import FixedNoiseBlock, Wheel, transform

MOJITO_PATH = os.environ.get("ENCHILADA_MOJITO_L1")
if not MOJITO_PATH:
    raise SystemExit(
        "set ENCHILADA_MOJITO_L1 to a Mojito-layout L1 file "
        "(in Jupyter: %env ENCHILADA_MOJITO_L1=/path/to/file.h5, then rerun this cell)"
    )
DURATION_S = float(os.environ.get("FORGE_DURATION_S", 86400.0))
FACTOR = int(os.environ.get("FORGE_FACTOR", 10))
DOMAIN = "frequency"  # the grid ForgeBlock renders on; see _check_residual
# The campaign's WDM home grid: this row count and the decimated window length
# divided by it must both be even integers (64 rows for the default day at
# factor 10 from a 4 Hz file). DOMAIN = "wdm" serves the block on this grid.
NUM_FREQUENCY_DIVISIONS = int(os.environ.get("FORGE_ROWS", 64))
CHANNELS = ("A", "E")
TRUTH = {}  # EDIT: injected parameter values, when you know them: {"amplitude": 1e-18}

window = load_window(
    MOJITO_PATH, factor=FACTOR, duration_s=DURATION_S, channels=CHANNELS
)
N = window.N
if not (
    NUM_FREQUENCY_DIVISIONS > 0
    and NUM_FREQUENCY_DIVISIONS % 2 == 0
    and N % (2 * NUM_FREQUENCY_DIVISIONS) == 0
):
    raise SystemExit(
        f"FORGE_ROWS={NUM_FREQUENCY_DIVISIONS} must be even and divide the "
        f"{N}-sample window (FORGE_DURATION_S={DURATION_S}, FORGE_FACTOR={FACTOR}) "
        "with an even quotient"
    )
# A white stand-in for the simulation's noise model, at the window's own level
# (for white native noise about 12 % below the in-band PSD: decimation rolls
# off the top fifth of the band; a loud signal in the window inflates it).
# The GB example's `FixedLISANoise().psd` (copy `examples/gb_model.py` beside
# this notebook; needs lisaanalysistools) is a TDI 1.5 sensitivity; Mojito's
# channels are TDI 2.0, so convert it before using it as `PSD`.
LEVEL = 2 * np.mean([window.channel_data[name].var() for name in CHANNELS]) / window.fs
PSD = white_psd(LEVEL)
residual = (
    transform(window, DOMAIN, num_frequency_divisions=NUM_FREQUENCY_DIVISIONS)
    if DOMAIN == "wdm"
    else transform(window, DOMAIN)
)
covariance = analytic_covariance(residual, PSD)

In [ ]:
def rms(data):
    return float(
        np.sqrt(np.mean([np.abs(data.channel_data[name]) ** 2 for name in CHANNELS]))
    )


print(f"data rms                      {rms(residual):.3e}")
wheel = Wheel(residual, num_frequency_divisions=NUM_FREQUENCY_DIVISIONS, random_seed=0)
# The PSD's covariance enters as a noise block, registered first so the signal
# blocks' starts receive it. It publishes on the residual's grid; the ledger
# keeps it there and the noise sum on the WDM home grid receives its translation.
wheel.add(FixedNoiseBlock("psd", covariance))
block = ForgeBlock(name="forge", frequency_hz=1e-3)  # EDIT: your block's configuration
wheel.add(block, data_domain=DOMAIN)
# A second signal block goes here, on whatever grid it renders on:
# wheel.add(OtherBlock(name="other"), data_domain="time")
# Every block receives the noise sum translated onto its own grid: white is exact
# on every grid; a coloured PSD arrives band-smoothed on the frequency grid and as
# its mean power, one flat variance, on the time grid.
print(f"residual rms after the starts {rms(wheel.residual()):.3e}")

In [ ]:
NUM_CYCLES = 200
WARM_UP = 0  # the placeholder draws the posterior exactly; an MCMC sampler needs some
chain = []


def collect(cycle_index, campaign):
    chain.append(dict(campaign.ledger[block.name].parameters))


wheel.run(NUM_CYCLES, on_cycle_complete=collect)
kept = chain[WARM_UP:]
for key, first in kept[0].items():
    if not isinstance(first, (int, float, np.number)):
        continue  # summarise numeric scalars only
    samples = np.asarray([parameters[key] for parameters in kept])
    print(
        f"posterior {key}: {samples.mean():+.3e} +/- {samples.std():.3e} "
        f"over {samples.size} cycles"
    )
    if key in TRUTH:
        offset = (samples.mean() - TRUTH[key]) / samples.std()
        print(
            f"truth {TRUTH[key]:+.3e}: the posterior mean is {offset:+.2f} sigma away"
        )
print(f"residual rms after the run    {rms(wheel.residual()):.3e}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None:
    keys = [
        key
        for key, value in chain[0].items()
        if isinstance(value, (int, float, np.number))
    ]
    figure, axes = plt.subplots(
        len(keys), 1, figsize=(8, 3 * len(keys)), sharex=True, squeeze=False
    )
    for axis, key in zip(axes[:, 0], keys, strict=True):
        axis.plot([parameters[key] for parameters in chain])
        axis.axvline(WARM_UP, color="k", linestyle=":")
        if key in TRUTH:
            axis.axhline(TRUTH[key], color="k", linestyle="--")
        axis.set_ylabel(key)
    axes[-1, 0].set_xlabel("cycle")
    plt.show()

## Next

- Replace the placeholder model in `block.py` and rerun both notebooks.
- Model the noise with a noise block (a block declaring `role = "noise"`),
  registered before the signal blocks whose starts need its covariance; blocks
  start in registration order. It publishes on the grid it registered on, the
  noise sum on the home grid receives its translation beside the PSD's
  `FixedNoiseBlock`, and every block receives the sum on its own grid; drop
  that block when yours models the whole noise.
- Run independent campaigns with fresh block instances and different seeds.
- Checkpointing and restarts are outside the forge.